# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [11]:
class matrix:
    def __init__(self, n=None, m=None, values=None):

        # If a list of lists is given
        if values is not None:
            cols = len(values[0])

            # Make sure every row has same number of columns
            for row in values:
                if len(row) != cols:
                    raise ValueError("All rows must have the same number of columns")

            self.n = len(values)
            self.m = cols

            # Copy the values so it doesn’t link to the original list
            self.data = []
            for row in values:
                self.data.append(list(row))

        # If n and m are given, create a zero matrix
        elif n is not None and m is not None:
            self.n = n
            self.m = m
            self.data = []
            for i in range(n):
                self.data.append([0.0] * m)

        # If nothing is given, create an empty matrix
        else:
            self.n = 0
            self.m = 0
            self.data = []

    def __getitem__(self, index):

        # This allows M[i, j]
        if isinstance(index, tuple):
            i, j = index
            return self.data[i][j]

        # This allows M[i][j]
        else:
            return self.data[index]

    def assign(self, other):

        # If assigning from another matrix
        if isinstance(other, matrix):
            if self.n != other.n or self.m != other.m:
                raise ValueError("Matrices must be the same size to assign")

            self.data = []
            for row in other.data:
                self.data.append(list(row))

        # If assigning from a list of lists
        elif isinstance(other, list):
            if len(other) != self.n or len(other[0]) != self.m:
                raise ValueError("Dimension of the lists doesn't match the matrix size")

            self.data = []
            for row in other:
                self.data.append(list(row))

In [12]:
# Make a 2x2 matrix filled with zeros
M_1 = matrix(n=2, m=2)
print("M_1 data:", M_1.data)

# Make another 2x2 matrix using actual numbers
M_2 = matrix(values=[[1, 2], [3, 4]])
print("M_2 data:", M_2.data)

# Copy values from M_2 into M_1
M_1.assign(M_2)
print("M_1 after assigning M_2:", M_1.data)


M_1 data: [[0.0, 0.0], [0.0, 0.0]]
M_2 data: [[1, 2], [3, 4]]
M_1 after assigning M_2: [[1, 2], [3, 4]]


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows.
    * Modify `__getitem__` implemented above to support slicing.
        

In [13]:
# Inheriting from the matrix class in Q1 so we keep previous methods
class matrix(matrix):

    def shape(self):
        # returns the size of the matrix as (rows, columns)
        return (self.n, self.m)

    def to_list(self):
        # returns a normal Python list of lists
        out = list()
        for row in self.data:
            out.append(list(row))
        return out

    def transpose(self):
        # swaps rows and columns
        out_data = list()
        for j in range(self.m):
            new_row = list()
            for i in range(self.n):
                new_row.append(self.data[i][j])
            out_data.append(new_row)
        return matrix(values=out_data)

    def row(self, n):
        # returns the nth row as a 1 x m matrix
        row_data = [self.data[n]]
        return matrix(values=row_data)

    def column(self, n):
        # returns the nth column as an n x 1 matrix
        col_data = list()
        for i in range(self.n):
            col_data.append([self.data[i][n]])
        return matrix(values=col_data)

    def block(self, n_0, n_1, m_0, m_1):
        # returns a smaller matrix using the given row and column range
        block_data = list()
        for i in range(m_0, m_1):
            new_row = self.data[i][n_0:n_1]
            block_data.append(new_row)
        return matrix(values=block_data)

    def __getitem__(self, index):
        # updated to support slicing
        if isinstance(index, tuple):
            i = index[0]
            j = index[1]

            # if both are integers, return a single value
            if isinstance(i, int) and isinstance(j, int):
                return self.data[i][j]

            rows = self.data[i]

            # if row is an integer but column is a slice
            if isinstance(i, int):
                return rows[j]

            # if rows are sliced, apply column slice to each row
            out = list()
            for row in rows:
                out.append(row[j])
            return out
        else:
            return self.data[index]

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [14]:
# They should not be indented inside the matrix class

def constant(n, m, c):
    # Create an n x m matrix filled with the float value c
    data = [[float(c)] * m for i in range(n)]
    return matrix(values=data)

def zeros(n, m):
    # Return an n x m matrix filled with 0.0
    return constant(n, m, 0.0)

def ones(n, m):
    # Return an n x m matrix filled with 1.0
    return constant(n, m, 1.0)

def eye(n):
    # Start with an n x n matrix filled with 0.0
    data = [[0.0] * n for i in range(n)]

    # Set the diagonal elements to 1.0
    for i in range(n):
        data[i][i] = 1.0

    return matrix(values=data)

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [15]:
class matrix(matrix):

    def scalarmul(self, c):
        # Multiply every element in the matrix by c
        out_data = list()
        for i in range(self.n):
            new_row = list()
            for j in range(self.m):
                new_row.append(self.data[i][j] * c)
            out_data.append(new_row)
        return matrix(values=out_data)

    def add(self, N):
        # Addition only works if both matrices are the same size
        if self.n != N.n or self.m != N.m:
            raise ValueError("Matrices must be the same size to add.")

        out_data = list()
        for i in range(self.n):
            new_row = list()
            for j in range(self.m):
                new_row.append(self.data[i][j] + N.data[i][j])
            out_data.append(new_row)
        return matrix(values=out_data)

    def sub(self, N):
        # Subtraction also requires the same dimensions
        if self.n != N.n or self.m != N.m:
            raise ValueError("Matrices must be the same size to subtract.")

        out_data = list()
        for i in range(self.n):
            new_row = list()
            for j in range(self.m):
                new_row.append(self.data[i][j] - N.data[i][j])
            out_data.append(new_row)
        return matrix(values=out_data)

    def mat_mult(self, N):
        # For matrix multiplication:
        # number of columns in self must equal number of rows in N
        if self.m != N.n:
            raise ValueError("Inner dimensions must match for matrix multiplication.")

        out_data = list()
        for i in range(self.n):
            new_row = list()
            for j in range(N.m):
                dot_product = 0.0
                for k in range(self.m):
                    dot_product += self.data[i][k] * N.data[k][j]
                new_row.append(dot_product)
            out_data.append(new_row)
        return matrix(values=out_data)

    def element_mult(self, N):
        # Element-wise multiplication needs same dimensions
        if self.n != N.n or self.m != N.m:
            raise ValueError("Matrices must be the same size for element-wise multiplication.")

        out_data = list()
        for i in range(self.n):
            new_row = list()
            for j in range(self.m):
                new_row.append(self.data[i][j] * N.data[i][j])
            out_data.append(new_row)
        return matrix(values=out_data)

    def equals(self, N):
        # First check if N is even a matrix
        if not isinstance(N, matrix):
            return False

        # Sizes must match
        if self.n != N.n or self.m != N.m:
            return False

        # Compare every element
        for i in range(self.n):
            for j in range(self.m):
                if self.data[i][j] != N.data[i][j]:
                    return False

        return True

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [16]:
class matrix(matrix):

    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, matrix):
            return self.mat_mult(other)
        else:
            raise ValueError("Can only multiply by a number or another matrix.")

    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        else:
            raise ValueError("Can only multiply by a number.")

    def __eq__(self, other):
        return self.equals(other)

    # '=' cannot be overloaded in Python.
    # Use M.assign(N) if you want to copy values instead of references.

6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [17]:
# Create three 2x2 matrices
A = matrix(values=[[1.0, 2.0], [3.0, 4.0]])
B = matrix(values=[[5.0, 6.0], [7.0, 8.0]])
C = matrix(values=[[2.0, 0.0], [1.0, 2.0]])

# Identity matrix
I = eye(2)

print("Testing Matrix Properties")

# 1. (AB)C = A(BC)
left_1 = (A * B) * C
right_1 = A * (B * C)
print("(AB)C == A(BC):", left_1 == right_1)

# 2. A(B + C) = AB + AC
left_2 = A * (B + C)
right_2 = (A * B) + (A * C)
print("A(B+C) == AB + AC:", left_2 == right_2)

# 3. AB ≠ BA
AB = A * B
BA = B * A
print("AB != BA:", not (AB == BA))

# 4. AI = A
left_4 = A * I
print("AI == A:", left_4 == A)

Testing Matrix Properties
(AB)C == A(BC): True
A(B+C) == AB + AC: True
AB != BA: True
AI == A: True
